# Kiểm tra dữ liệu đầu vào — Issue #2

## 1. Mục tiêu và nguồn dữ liệu

Dataset validation kiểm tra xem tệp đầu vào có đúng cấu trúc và đáp ứng các quy tắc chất lượng đã định trước khi dùng cho EDA hoặc huấn luyện. Bước này giúp phát hiện sớm tệp sai, thiếu cột hoặc giá trị khó xử lý ở các bước sau.

Bộ [Kaggle Playground Series S4E2](https://www.kaggle.com/competitions/playground-series-s4e2/data) gồm `train.csv` có nhãn dùng trong phần này, `test.csv` không có nhãn dùng để dự đoán khi dự thi, và `sample_submission.csv` minh họa định dạng nộp bài.

## 2. Schema và bảy nhãn

Số dòng giúp phát hiện tệp thiếu hoặc sai phiên bản. Tên cột, số cột và kiểu giá trị bảo đảm mỗi trường được diễn giải đúng. Schema dự kiến có 18 cột, gồm `id` là số nguyên, 16 đặc trưng số hoặc category, và `NObeyesdad` là nhãn đích. `id` chỉ dùng để định danh bản ghi.

Bảy nhãn biểu thị các nhóm cân nặng trong bộ dữ liệu:

- `Insufficient_Weight` — thiếu cân.
- `Normal_Weight` — cân nặng bình thường.
- `Overweight_Level_I` và `Overweight_Level_II` — thừa cân mức I và II.
- `Obesity_Type_I`, `Obesity_Type_II` và `Obesity_Type_III` — béo phì loại I, II và III.

## 3. Kiểm tra chất lượng

- **Missing values:** đếm ô trống và các dấu hiệu như `NA` hoặc `NULL` theo từng cột. Cột không có mặt được báo là lỗi schema.
- **Duplicate:** ID trùng là ID hợp lệ lặp lại. Trùng toàn dòng so khớp tất cả các trường. Trùng khi bỏ ID so khớp nội dung còn lại, kể cả nhãn. Mỗi phép đếm ghi số lần lặp sau lần xuất hiện đầu tiên.
- **Bất thường:** cột số được kiểm tra khả năng đọc thành số hữu hạn và so với khoảng sàng lọc. Cột category được so khớp chính xác với danh sách cho phép.

## 4. Cách chạy và đọc báo cáo

Dùng kernel Python 3.11 của dự án. Từ thư mục gốc, chạy `python -m src.data_loader` để xuất hai bảng vào `outputs/tables/`. Mã thoát 0 nghĩa là mọi phép kiểm tra đạt, 1 nghĩa là đã xuất báo cáo có mục được đánh dấu, và 2 nghĩa là lệnh không hoàn tất.

- **`dataset_validation_checks.csv`:** `check` là tên phép kiểm tra, `column` là cột liên quan (trống nếu áp dụng cho cả bảng), `count` là số quan sát hoặc số vi phạm tùy phép kiểm tra, `criterion` là quy tắc, và `passed` cho biết quy tắc có đạt hay không. Mục không xác định có thể để trống số đếm và trạng thái. Ví dụ, `row_count` có `count` là số dòng quan sát, còn `missing` có `count` là số ô thiếu.
- **`dataset_validation_labels.csv`:** `label` là nhãn, `status` phân biệt nhãn dự kiến, nhãn lạ, nhãn thiếu hoặc cột nhãn không có mặt, và `count` là số dòng tương ứng. Khi schema hợp lệ, đối chiếu tổng của bảy nhãn dự kiến với tổng số dòng train.

## 5. Diễn giải kết quả đã lưu

Tệp local có **20.758 dòng, 18 cột và 16 đặc trưng**. Cả bảy nhãn đều xuất hiện và tổng số đếm bằng 20.758. Theo các quy tắc hiện tại, số ô thiếu, số lần trùng ở cả ba phép kiểm tra và số giá trị bị đánh dấu bất thường đều bằng 0.

Các khoảng số chỉ là quy tắc sàng lọc. Schema hợp lệ vẫn có thể đi kèm nhãn sai, thiên lệch hoặc các bản ghi gần giống nhau. Phép trùng khi bỏ ID vẫn giữ nhãn, nên các dòng cùng đặc trưng nhưng khác nhãn cần được xem xét riêng trước khi chia tập.


In [1]:
from hashlib import sha256
from pathlib import Path
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "src" / "data_loader.py").is_file():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_loader import read_csv, validate_dataset

source = root / "data" / "train.csv"
if not source.is_file():
    print("Pending: download the official Kaggle archive and place train.csv in data/.")
else:
    before = sha256(source.read_bytes()).hexdigest()
    columns, rows = read_csv(source)
    checks, labels = validate_dataset(columns, rows)
    after = sha256(source.read_bytes()).hexdigest()
    if before != after:
        raise RuntimeError("The input CSV changed during validation.")

    print(f"Input SHA-256: {after}")
    print("Columns:", ", ".join(columns))
    print("Check | Column | Count | Criterion | Passed")
    for check, column, count, criterion, passed in checks:
        value = "unavailable" if count is None else count
        print(f"{check} | {column} | {value} | {criterion} | {passed}")
    print("Label | Status | Count")
    for label, status, count in labels:
        print(f"{label} | {status} | {count}")


Input SHA-256: fec835cc642ffbc3274a026abfe465d0aaceedfe2f7b35d987029f676d7dae6d
Columns: id, Gender, Age, Height, Weight, family_history_with_overweight, FAVC, FCVC, NCP, CAEC, SMOKE, CH2O, SCC, FAF, TUE, CALC, MTRANS, NObeyesdad
Check | Column | Count | Criterion | Passed
row_count |  | 20758 | Expected 20758 | True
column_count |  | 18 | Expected 18 | True
feature_count |  | 16 | Expected 16 | True
missing | id | 0 | Blank, NA, N/A, NaN or NULL | True
invalid_id | id | 0 | Nonnegative decimal integer, at most 18 digits | True
duplicate_id | id | 0 | Repeated valid integer ID, excluding first occurrence | True
missing | Gender | 0 | Blank, NA, N/A, NaN or NULL | True
unexpected_category | Gender | 0 | Exact match to Female | Male | True
missing | Age | 0 | Blank, NA, N/A, NaN or NULL | True
invalid_numeric | Age | 0 | Finite number required | True
out_of_range | Age | 0 | Inclusive screening range [0, 120] | True
missing | Height | 0 | Blank, NA, N/A, NaN or NULL | True
invalid_numeri